# Misure esterne di complessità dei prodotti (indipendenti dal commercio)

Porta sui codici HS 2002 (4 e 6 cifre) misure costruite **senza** dati di commercio paese-prodotto:

| misura | livello | fonte |
|---|---|---|
| classe tecnologica di Lall (2000) | prodotto (SITC 3 cifre) | UNCTAD |
| Advanced Technology Products (ATP) | prodotto (HS 10 cifre) | US Census |
| intensità di R&D (R&D / valore aggiunto) | settore (ISIC Rev.4) | OECD, Galindo-Rueda & Verger (2016) |
| valore aggiunto per addetto, quota di lavoratori non di produzione (skill), salario medio, capitale per addetto | industria manifatturiera (NAICS 6 cifre) | NBER-CES |

Ponte: concordanza ufficiale US Census 2005 (Schedule B, HS 2002 a 10 cifre → NAICS 2002, SITC Rev.3, flag ATP).
Un codice HS a 4/6 cifre raccoglie più righe a 10 cifre: le variabili categoriche prendono il valore più frequente
(la colonna `purezza_*` dice quanta parte delle righe concorda), la quota ATP è la frazione di righe ATP.

File grezzi e fonti: `data/raw/complessita_prodotti/README.md`.
Output: `results/econometria/misure_esterne/product/misure_esterne_hs{4,6}.parquet` (vedi `src/storage.py`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.storage import external_product_path

## Configurazione

In [ ]:
RAW = ROOT / "data" / "raw" / "complessita_prodotti"
NBER_YEARS = range(2002, 2007)        # media sugli anni di validita' di HS 2002, per ridurre il rumore annuale
LEVELS = [6, 4]

# Lall (2000) -> classe ordinale
LALL_RANK = {"LDC01": 0, "LDC02": 1, "LDC03": 1, "LDC04": 2, "LDC05": 2,
             "LDC06": 3, "LDC07": 3, "LDC08": 3, "LDC09": 4, "LDC10": 4}
LALL_NAME = {0: "primari", 1: "basati su risorse", 2: "bassa tecnologia", 3: "media tecnologia", 4: "alta tecnologia"}

## Corrispondenza NAICS → gruppi OECD (ISIC Rev.4)

Costruita a mano a 3–4 cifre NAICS seguendo le definizioni dei gruppi della Tabella 1 di Galindo-Rueda & Verger (2016).

In [ ]:
def naics_to_oecd(n):
    if not isinstance(n, str) or len(n) < 3:
        return np.nan
    s2, s3, s4 = n[:2], n[:3], n[:4]
    if s2 == "11": return "01-03"
    if s2 == "21": return "05-09"
    if s4 == "3254": return "21"                                      # farmaceutica
    if n in ("332992", "332993", "332994", "332995"): return "2521"   # armi e munizioni
    if s4 in ("3361", "3362", "3363"): return "29"                    # autoveicoli
    if s4 == "3364": return "3031"                                    # aerospazio
    if s4 == "3366": return "3011"                                    # navi e imbarcazioni
    if s4 in ("3365", "3369"): return "30X1"                          # ferrovie, altri mezzi
    if s4 == "3391": return "3251"                                    # strumenti medici
    return {"311": "10-12", "312": "10-12", "313": "13", "314": "13", "315": "14", "316": "15",
            "321": "16", "322": "17", "323": "18", "324": "19", "325": "20", "326": "22",
            "327": "23", "331": "24", "332": "25X1", "333": "28", "334": "26", "335": "27",
            "337": "31", "339": "32X1"}.get(s3, np.nan)

## Dati

In [ ]:
conc = pd.read_csv(RAW / "census_expconcord05.csv", dtype=str)
conc = conc[conc["COMMODITY"].str.fullmatch(r"\d{10}") & ~conc["COMMODITY"].str.startswith("98")]   # no codici speciali
conc["sitc3"] = conc["SITC"].str[:3]
conc["atp"] = (conc["HITECH"].fillna("00") != "00").astype(float)
lall = pd.read_csv(RAW / "lall_2000_sitc3.csv", dtype=str).set_index("sitc3")["lall_code"]
conc["lall_code"] = conc["sitc3"].map(lall)
conc["oecd_group"] = conc["NAICS"].map(naics_to_oecd)

rd = pd.read_csv(RAW / "oecd_rd_intensity_isic4.csv", dtype={"oecd_group": str}).set_index("oecd_group")

nb = pd.read_csv(RAW / "nberces5818v1_n1997.csv", dtype={"naics": str})
nb = nb[nb["year"].isin(NBER_YEARS)].groupby("naics")[["vadd", "emp", "prode", "pay", "cap"]].mean()
nber = pd.DataFrame({
    "va_per_addetto": nb["vadd"] / nb["emp"],               # migliaia di $ per addetto
    "quota_skill":    1 - nb["prode"] / nb["emp"],          # quota di lavoratori non di produzione
    "salario_medio":  nb["pay"] / nb["emp"],
    "capitale_per_addetto": nb["cap"] / nb["emp"],
})
print(f"concordanza: {len(conc)} codici HS10, {conc.NAICS.notna().mean():.1%} con NAICS, "
      f"{conc.lall_code.notna().mean():.1%} con classe Lall; NBER: {len(nber)} industrie")

## Aggregazione ai codici HS a 4 e 6 cifre

In [ ]:
def mode_and_purity(s):
    s = s.dropna()
    if s.empty:
        return pd.Series({"v": np.nan, "p": np.nan})
    vc = s.value_counts()
    return pd.Series({"v": vc.index[0], "p": vc.iloc[0] / vc.sum()})

def build(level):
    g = conc.assign(hs=conc["COMMODITY"].str[:level]).groupby("hs")
    out = pd.DataFrame(index=g.size().index)
    for col, name in (("NAICS", "naics"), ("lall_code", "lall"), ("oecd_group", "oecd_group")):
        mp = g[col].apply(mode_and_purity).unstack()
        out[name], out[f"purezza_{name}"] = mp["v"], mp["p"].astype(float)
    out["quota_atp"] = g["atp"].mean()
    out["lall_classe"] = out["lall"].map(LALL_RANK)
    out["rd_intensity"] = out["oecd_group"].map(rd["rd_intensity_pct_gva"])
    out = out.join(nber, on="naics")
    out.index.name = f"hs{level}"
    return out

external_product_path(4).parent.mkdir(parents=True, exist_ok=True)
for level in LEVELS:
    t = build(level)
    t.to_parquet(external_product_path(level))
    print(f"HS{level}: {len(t)} codici | copertura: Lall {t.lall_classe.notna().mean():.1%}, "
          f"R&D {t.rd_intensity.notna().mean():.1%}, NBER {t.va_per_addetto.notna().mean():.1%}")
t.head()

## Copertura sui prodotti delle tue matrici

In [ ]:
for level in LEVELS:
    C = pd.read_parquet(ROOT / "data" / "HS02_all_years" / f"{level}_digits" / "alligned" / "binarized" / "C" / f"C_{level}_digits_HS02.parquet")
    t = pd.read_parquet(external_product_path(level))
    prods = pd.Index(C.columns)
    print(f"HS{level}: {prods.isin(t.index).mean():.1%} dei {len(prods)} prodotti presenti nella concordanza; "
          f"con classe Lall {t.reindex(prods).lall_classe.notna().mean():.1%}, "
          f"con NBER {t.reindex(prods).va_per_addetto.notna().mean():.1%}")